# Fetch AWS VM Costs (All Tiers)

This notebook fetches **ALL** AWS EC2 pricing tiers sequentially to avoid rate limiting:
1. On-Demand pricing
2. Spot pricing (90-day average)
3. Reserved 1-Year pricing (No Upfront, Partial Upfront, All Upfront)
4. Reserved 3-Year pricing (No Upfront, Partial Upfront, All Upfront)

**Data Source:** AWS Pricing API & EC2 Spot Price History API  
**Output:** `vm_costs` table with cloud = 'AWS'

**New column:** `payment_option` - tracks upfront payment type for reserved instances


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_INSTANCE_RATES = f"{CATALOG}.{SCHEMA}.instance_rates"
TABLE_VM_COSTS = f"{CATALOG}.{SCHEMA}.vm_costs"
TABLE_SKU_REGION_MAPPING = f"{CATALOG}.{SCHEMA}.sku_region_mapping"

# Load AWS regions from sku_region_mapping (SOURCE OF TRUTH from dbu_prices)
# This ensures we fetch VM costs for ALL regions that have DBU pricing
regions_df = spark.sql(f"""
    SELECT DISTINCT region_code 
    FROM {TABLE_SKU_REGION_MAPPING} 
    WHERE UPPER(cloud) = 'AWS'
""").toPandas()

AWS_REGIONS = regions_df['region_code'].tolist()

# AWS region code to location name mapping (for Pricing API)
AWS_REGION_TO_LOCATION = {
    'us-east-1': 'US East (N. Virginia)',
    'us-east-2': 'US East (Ohio)',
    'us-west-1': 'US West (N. California)',
    'us-west-2': 'US West (Oregon)',
    'ca-central-1': 'Canada (Central)',
    'eu-west-1': 'EU (Ireland)',
    'eu-west-2': 'EU (London)',
    'eu-west-3': 'EU (Paris)',
    'eu-central-1': 'EU (Frankfurt)',
    'eu-north-1': 'EU (Stockholm)',
    'ap-northeast-1': 'Asia Pacific (Tokyo)',
    'ap-northeast-2': 'Asia Pacific (Seoul)',
    'ap-northeast-3': 'Asia Pacific (Osaka)',
    'ap-southeast-1': 'Asia Pacific (Singapore)',
    'ap-southeast-2': 'Asia Pacific (Sydney)',
    'ap-southeast-3': 'Asia Pacific (Jakarta)',
    'ap-south-1': 'Asia Pacific (Mumbai)',
    'sa-east-1': 'South America (Sao Paulo)',
    'me-central-1': 'Middle East (UAE)',
}

print(f"✅ Config: {CATALOG}.{SCHEMA}")
print(f"📍 Loaded {len(AWS_REGIONS)} AWS regions from sku_region_mapping")
print(f"   Regions: {AWS_REGIONS}")


In [ ]:
%pip install boto3 -q

import boto3
import json
import os
import time
from datetime import datetime, timedelta
import pandas as pd

# Load AWS credentials from secret scope
SECRET_SCOPE = "lakemeter-credentials"
os.environ['AWS_ACCESS_KEY_ID'] = dbutils.secrets.get(SECRET_SCOPE, "aws-access-key-id")
os.environ['AWS_SECRET_ACCESS_KEY'] = dbutils.secrets.get(SECRET_SCOPE, "aws-secret-access-key")

print(f"✅ AWS credentials loaded from secret scope")

# Get instance types
instance_types_df = spark.sql(f"""
    SELECT DISTINCT instance_type FROM {TABLE_INSTANCE_RATES} WHERE cloud = 'AWS'
""").toPandas()
INSTANCE_TYPES = instance_types_df['instance_type'].tolist()

print(f"📋 Will fetch pricing for {len(INSTANCE_TYPES)} instance types")


In [ ]:
# ============================================================
# PART 1: ON-DEMAND PRICING
# ============================================================
print("=" * 60)
print("💰 PART 1: FETCHING ON-DEMAND PRICES")
print("=" * 60)

pricing_client = boto3.client('pricing', region_name='us-east-1')
all_prices = []

def get_ondemand_price(instance_type: str, region: str) -> float:
    location = AWS_REGION_TO_LOCATION.get(region)
    if not location: return None
    try:
        response = pricing_client.get_products(
            ServiceCode='AmazonEC2',
            Filters=[
                {'Type': 'TERM_MATCH', 'Field': 'instanceType', 'Value': instance_type},
                {'Type': 'TERM_MATCH', 'Field': 'location', 'Value': location},
                {'Type': 'TERM_MATCH', 'Field': 'tenancy', 'Value': 'Shared'},
                {'Type': 'TERM_MATCH', 'Field': 'operatingSystem', 'Value': 'Linux'},
                {'Type': 'TERM_MATCH', 'Field': 'preInstalledSw', 'Value': 'NA'},
                {'Type': 'TERM_MATCH', 'Field': 'capacitystatus', 'Value': 'Used'},
            ],
            MaxResults=1
        )
        if response['PriceList']:
            product = json.loads(response['PriceList'][0])
            for term in product.get('terms', {}).get('OnDemand', {}).values():
                for pd in term.get('priceDimensions', {}).values():
                    price = float(pd['pricePerUnit'].get('USD', 0))
                    if price > 0: return price
    except Exception as e:
        pass
    return None

for region in AWS_REGIONS:
    print(f"📍 {region}", end=" ")
    count = 0
    for inst in INSTANCE_TYPES:
        price = get_ondemand_price(inst, region)
        if price:
            all_prices.append({
                "cloud": "AWS", "region": region, "instance_type": inst,
                "pricing_tier": "on_demand", "payment_option": None,
                "cost_per_hour": round(price, 6),
                "currency": "USD", "source": "AWS Pricing API",
                "fetched_at": datetime.utcnow().isoformat()
            })
            count += 1
        time.sleep(0.05)  # Rate limiting
    print(f"✅ {count}")

print(f"\n✅ On-Demand: {len([p for p in all_prices if p['pricing_tier'] == 'on_demand'])} prices")


In [ ]:
# ============================================================
# PART 2: SPOT PRICING (90-day average)
# ============================================================
print("\n" + "=" * 60)
print("💰 PART 2: FETCHING SPOT PRICES (90-day avg)")
print("=" * 60)

LOOKBACK_DAYS = 90

def get_spot_avg(ec2_client, instance_types: list) -> dict:
    end_time = datetime.utcnow()
    start_time = end_time - timedelta(days=LOOKBACK_DAYS)
    prices = {}
    
    for i in range(0, len(instance_types), 10):
        batch = instance_types[i:i+10]
        try:
            paginator = ec2_client.get_paginator('describe_spot_price_history')
            for page in paginator.paginate(
                InstanceTypes=batch, ProductDescriptions=['Linux/UNIX'],
                StartTime=start_time, EndTime=end_time
            ):
                for rec in page['SpotPriceHistory']:
                    inst = rec['InstanceType']
                    if inst not in prices: prices[inst] = []
                    prices[inst].append(float(rec['SpotPrice']))
        except: pass
    
    return {k: sum(v)/len(v) for k, v in prices.items() if v}

for region in AWS_REGIONS:
    print(f"📍 {region}", end=" ")
    ec2_client = boto3.client('ec2', region_name=region)
    spot_prices = get_spot_avg(ec2_client, INSTANCE_TYPES)
    
    for inst, price in spot_prices.items():
        all_prices.append({
            "cloud": "AWS", "region": region, "instance_type": inst,
            "pricing_tier": "spot", "payment_option": None,
            "cost_per_hour": round(price, 6),
            "currency": "USD", "source": f"AWS EC2 API ({LOOKBACK_DAYS}-day avg)",
            "fetched_at": datetime.utcnow().isoformat()
        })
    print(f"✅ {len(spot_prices)}")

print(f"\n✅ Spot: {len([p for p in all_prices if p['pricing_tier'] == 'spot'])} prices")


In [ ]:
# ============================================================
# PART 3: RESERVED PRICING (1Y and 3Y - All Payment Options)
# ============================================================
print("\n" + "=" * 60)
print("💰 PART 3: FETCHING RESERVED PRICES (1Y & 3Y - All Upfront Variants)")
print("=" * 60)

# Payment options to fetch
PAYMENT_OPTIONS = ["No Upfront", "Partial Upfront", "All Upfront"]

def get_reserved_price(instance_type: str, region: str, lease_length: str, purchase_option: str) -> float:
    """Get reserved instance price for specific payment option."""
    location = AWS_REGION_TO_LOCATION.get(region)
    if not location: return None
    try:
        response = pricing_client.get_products(
            ServiceCode='AmazonEC2',
            Filters=[
                {'Type': 'TERM_MATCH', 'Field': 'instanceType', 'Value': instance_type},
                {'Type': 'TERM_MATCH', 'Field': 'location', 'Value': location},
                {'Type': 'TERM_MATCH', 'Field': 'tenancy', 'Value': 'Shared'},
                {'Type': 'TERM_MATCH', 'Field': 'operatingSystem', 'Value': 'Linux'},
                {'Type': 'TERM_MATCH', 'Field': 'preInstalledSw', 'Value': 'NA'},
                {'Type': 'TERM_MATCH', 'Field': 'capacitystatus', 'Value': 'Used'},
                {'Type': 'TERM_MATCH', 'Field': 'termType', 'Value': 'Reserved'},
                {'Type': 'TERM_MATCH', 'Field': 'LeaseContractLength', 'Value': lease_length},
                {'Type': 'TERM_MATCH', 'Field': 'PurchaseOption', 'Value': purchase_option},
            ],
            MaxResults=10
        )
        for pj in response.get('PriceList', []):
            product = json.loads(pj)
            for term in product.get('terms', {}).get('Reserved', {}).values():
                # For Partial/All Upfront, we need to calculate effective hourly rate
                # Effective hourly = (upfront / hours_in_term) + hourly_rate
                upfront = 0
                hourly = 0
                hours_in_term = 8760 if lease_length == '1yr' else 26280  # 1yr or 3yr
                
                for pd in term.get('priceDimensions', {}).values():
                    if 'Quantity' in pd.get('unit', ''):
                        # This is the upfront payment
                        upfront = float(pd['pricePerUnit'].get('USD', 0))
                    elif 'Hrs' in pd.get('unit', ''):
                        # This is the hourly rate
                        hourly = float(pd['pricePerUnit'].get('USD', 0))
                
                effective_hourly = (upfront / hours_in_term) + hourly
                if effective_hourly > 0:
                    return effective_hourly
    except: 
        pass
    return None

# Fetch all combinations
for lease, tier in [('1yr', 'reserved_1y'), ('3yr', 'reserved_3y')]:
    print(f"\n📅 {lease.upper()} Reserved:")
    for payment_opt in PAYMENT_OPTIONS:
        print(f"\n  💳 {payment_opt}:")
        for region in AWS_REGIONS:
            print(f"    📍 {region}", end=" ")
            count = 0
            for inst in INSTANCE_TYPES:
                price = get_reserved_price(inst, region, lease, payment_opt)
                if price:
                    all_prices.append({
                        "cloud": "AWS", "region": region, "instance_type": inst,
                        "pricing_tier": tier, 
                        "payment_option": payment_opt.lower().replace(" ", "_"),
                        "cost_per_hour": round(price, 6),
                        "currency": "USD", 
                        "source": f"AWS Pricing API ({lease} {payment_opt})",
                        "fetched_at": datetime.utcnow().isoformat()
                    })
                    count += 1
                time.sleep(0.05)  # Rate limiting
            print(f"✅ {count}")

# Summary
from collections import Counter
reserved_counts = Counter((p['pricing_tier'], p['payment_option']) for p in all_prices if p['pricing_tier'].startswith('reserved'))
print(f"\n✅ Reserved pricing summary:")
for (tier, payment), count in sorted(reserved_counts.items()):
    print(f"   {tier} ({payment}): {count}")


In [ ]:
# ============================================================
# SAVE ALL PRICES
# ============================================================
print("\n" + "=" * 60)
print("💾 SAVING ALL AWS PRICES")
print("=" * 60)

df = pd.DataFrame(all_prices)
df['updated_at'] = datetime.utcnow()

print(f"\n📊 Total: {len(df)} prices")
print(f"   - On-Demand: {len(df[df['pricing_tier'] == 'on_demand'])}")
print(f"   - Spot: {len(df[df['pricing_tier'] == 'spot'])}")

# Reserved breakdown by payment option
for tier in ['reserved_1y', 'reserved_3y']:
    tier_df = df[df['pricing_tier'] == tier]
    print(f"   - {tier}:")
    for opt in ['no_upfront', 'partial_upfront', 'all_upfront']:
        count = len(tier_df[tier_df['payment_option'] == opt])
        print(f"       {opt}: {count}")

display(df.head(20))

if len(df) > 0:
    spark_df = spark.createDataFrame(df)
    if spark.catalog.tableExists(TABLE_VM_COSTS):
        spark.sql(f"DELETE FROM {TABLE_VM_COSTS} WHERE cloud = 'AWS'")
        print(f"🗑️ Deleted existing AWS records")
    spark_df.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLE_VM_COSTS)
    print(f"✅ Saved {len(df)} rows to {TABLE_VM_COSTS}")


In [ ]:
# Verify
display(spark.sql(f"""
    SELECT cloud, pricing_tier, payment_option, COUNT(*) as count,
           ROUND(MIN(cost_per_hour), 4) as min_cost,
           ROUND(AVG(cost_per_hour), 4) as avg_cost,
           ROUND(MAX(cost_per_hour), 4) as max_cost
    FROM {TABLE_VM_COSTS}
    WHERE cloud = 'AWS'
    GROUP BY cloud, pricing_tier, payment_option
    ORDER BY pricing_tier, payment_option
"""))
